### Removendo MultiLabel

In [1]:
import os
import shutil
import hashlib
import cv2
from collections import defaultdict

def remove_visual_duplicates_safe(
    export_dir="../dataset/export", 
    clean_export_dir="../dataset/clean_export",
    log_file="../dataset/duplicatas_ignoradas.txt"
):
    if not os.path.exists(export_dir):
        raise FileNotFoundError(f"A pasta {export_dir} não foi encontrada.")

    if os.path.exists(clean_export_dir):
        shutil.rmtree(clean_export_dir)
    os.makedirs(clean_export_dir)
    
    # Dicionário para agrupar imagens pelo seu hash
    hash_to_files = defaultdict(list)

    # Lista apenas as imagens na pasta export
    image_files = [f for f in os.listdir(export_dir) if f.lower().endswith(('.jpeg'))]
    
    # Mapea todos os Hashes silenciosamente
    for img_name in image_files:
        img_path = os.path.join(export_dir, img_name)
        
        img = cv2.imread(img_path)
        if img is not None:
            # Gera o Hash MD5 com base nos bytes puros da imagem
            img_hash = hashlib.md5(img.tobytes()).hexdigest()
            hash_to_files[img_hash].append(img_name)

    # Copia únicas e gera TXT das duplicatas 
    unicas_copiadas = 0
    duplicatas_ignoradas = 0

    # Abre o arquivo TXT para escrita
    with open(log_file, 'w', encoding='utf-8') as f:
        f.write("Relatório de Imagens Duplicadas (Ignoradas)\n")
        f.write("=" * 50 + "\n\n")

        for img_hash, file_list in hash_to_files.items():
            if len(file_list) > 1:
                # É uma duplicata (2 ou mais imagens idênticas)
                duplicatas_ignoradas += len(file_list)
                f.write(f"Grupo de imagens idênticas (Hash: {img_hash}):\n")
                for name in file_list:
                    f.write(f" -> {name}\n")
                f.write("\n")
            else:
                # É uma imagem única, fazemos a cópia
                img_name = file_list[0]
                img_path = os.path.join(export_dir, img_name)
                xml_name = os.path.splitext(img_name)[0] + ".xml"
                xml_path = os.path.join(export_dir, xml_name)
                
                shutil.copy2(img_path, os.path.join(clean_export_dir, img_name))
                if os.path.exists(xml_path):
                    shutil.copy2(xml_path, os.path.join(clean_export_dir, xml_name))
                
                unicas_copiadas += 1

    print("-" * 60)
    print("Resumo da Filtragem Segura:")
    print(f"Imagens únicas copiadas para '{clean_export_dir}': {unicas_copiadas}")
    print(f"Imagens duplicadas ignoradas: {duplicatas_ignoradas}")
    print(f"Relatório detalhado das duplicatas salvo em: '{log_file}'")
    print("-" * 60)

if __name__ == "__main__":
    remove_visual_duplicates_safe()

------------------------------------------------------------
Resumo da Filtragem Segura:
Imagens únicas copiadas para '../dataset/clean_export': 1426
Imagens duplicadas ignoradas: 134
Relatório detalhado das duplicatas salvo em: '../dataset/duplicatas_ignoradas.txt'
------------------------------------------------------------


### Recorte

In [2]:
import os
import json
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import pandas as pd

def process_rocole_xml_crop(
    annotations_dir="../dataset/Annotations",
    export_dir="../dataset/clean_export",
    output_dir="../dataset/preprocess_imgs"
):
    # 1. Estrutura de pastas de saída
    path_healthy = os.path.join(output_dir, "healthy")
    path_rust = os.path.join(output_dir, "rust")
    path_mite = os.path.join(output_dir, "red_spider_mite")

    for path in [path_healthy, path_rust, path_mite]:
        os.makedirs(path, exist_ok=True)

    print("Estrutura de diretórios configurada:")
    print(f" - {path_healthy}\n - {path_rust}\n - {path_mite}\n")

    # 2. Carregamento do CSV de anotações
    csv_file = os.path.join(annotations_dir, "RoCoLE-csv.csv")
    if not os.path.exists(csv_file):
        csv_files = [f for f in os.listdir(annotations_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {annotations_dir}")
        csv_file = os.path.join(annotations_dir, csv_files[0])

    try:
        df = pd.read_csv(csv_file, quotechar='"', skipinitialspace=True)
    except Exception as e:
        print(f"[AVISO] Falha na leitura padrão. Erro: {e}")
        df = pd.read_csv(csv_file, sep=None, engine='python')

    print(f"Arquivo CSV carregado: {os.path.basename(csv_file)}")
    print(f"Total de registros na planilha: {len(df)}\n")

    success_count = 0
    error_count = 0

    # 3. Iteração pelos registros
    for row in df.itertuples():
        row_id = str(getattr(row, 'ID', '')).strip()
        if not row_id or row_id.lower() == 'nan':
            continue

        img_name = f"{row_id}.jpeg"
        xml_name = f"{row_id}.xml"
        
        img_path = os.path.join(export_dir, img_name)
        xml_path = os.path.join(export_dir, xml_name)

        if not os.path.exists(img_path) or not os.path.exists(xml_path):
            print(f"[AVISO] Arquivo(s) ausente(s) para o ID: {row_id}")
            error_count += 1
            continue

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is None:
            print(f"[ERRO] Falha ao carregar a imagem: {img_path}")
            error_count += 1
            continue

        h, w = img.shape[:2]

        # Leitura e parsing do arquivo XML com captura específica de erro
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            size_elem = root.find('size')
            if size_elem is not None:
                xml_w = int(size_elem.find('width').text) if size_elem.find('width') is not None else w
                xml_h = int(size_elem.find('height').text) if size_elem.find('height') is not None else h
            else:
                xml_h, xml_w = h, w
        except ET.ParseError as err:
            print(f"[ERRO] Falha ao analisar o XML ({xml_name}): {err}")
            error_count += 1
            continue

        # Redimensionar se as dimensões do XML divergirem da imagem real
        if (h != xml_h) or (w != xml_w):
            img = cv2.resize(img, (xml_w, xml_h))
            h, w = xml_h, xml_w

        # Criar máscara preta
        mask = np.zeros((h, w), dtype=np.uint8)
        polygons_found = False

        # Extrair polígonos de todos os objetos no XML
        for obj in root.findall('object'):
            polygon_elem = obj.find('polygon')
            if polygon_elem is not None:
                coords = {}
                for child in polygon_elem:
                    tag = child.tag.strip()
                    val = float(child.text.strip()) if child.text else 0.0
                    
                    if tag.startswith('x') and tag[1:].isdigit():
                        coords.setdefault(int(tag[1:]), {})['x'] = int(val)
                    elif tag.startswith('y') and tag[1:].isdigit():
                        coords.setdefault(int(tag[1:]), {})['y'] = int(val)

                pts = [[coords[i]['x'], coords[i]['y']] for i in sorted(coords.keys()) if 'x' in coords[i] and 'y' in coords[i]]

                if pts:
                    cv2.fillPoly(mask, np.array([pts], dtype=np.int32), 255)
                    polygons_found = True

        # Aplica a máscara apenas se encontrou polígonos
        cropped_img = cv2.bitwise_and(img, img, mask=mask) if polygons_found else img.copy()

        # Extração direta e simplificada do JSON
        label_raw = getattr(row, 'Label', '{}')
        classification = "healthy"
        
        try:
            label_data = json.loads(label_raw) if isinstance(label_raw, str) else label_raw
            
            if 'classification' in label_data:
                classification = label_data['classification']
            elif 'Leaf' in label_data and isinstance(label_data['Leaf'], list) and len(label_data['Leaf']) > 0:
                leaf_data = label_data['Leaf'][0]
                classification = leaf_data.get('classification', leaf_data.get('state', 'healthy'))
        except (json.JSONDecodeError, TypeError):
            pass

        classification = classification.lower()

        # Determinar pasta de destino
        if 'rust' in classification:
            dest_dir = path_rust
        elif 'red_spider_mite' in classification:
            dest_dir = path_mite
        else:
            dest_dir = path_healthy

        save_path = os.path.join(dest_dir, img_name)
        cv2.imwrite(save_path, cropped_img)
        success_count += 1

    print(f"\n[SUCESSO] Total de imagens processadas via XML e salvas: {success_count}")
    print(f"[AVISO] Falhas/Arquivos não encontrados: {error_count}")

if __name__ == "__main__":
    process_rocole_xml_crop()

Estrutura de diretórios configurada:
 - ../dataset/preprocess_imgs\healthy
 - ../dataset/preprocess_imgs\rust
 - ../dataset/preprocess_imgs\red_spider_mite

Arquivo CSV carregado: RoCoLE-csv.csv
Total de registros na planilha: 1560

[AVISO] Arquivo(s) ausente(s) para o ID: cjvnxy9x6pd3p083891gbwrk2
[AVISO] Arquivo(s) ausente(s) para o ID: cjvnxyv9gpioo0866ich4sq0b
[AVISO] Arquivo(s) ausente(s) para o ID: cjvnxywnfpipp0866io18ls48
[AVISO] Arquivo(s) ausente(s) para o ID: cjvnxzeq5piy20866kup966wu
[AVISO] Arquivo(s) ausente(s) para o ID: cjvny0onjpjjg0866oj2x9n1u
[AVISO] Arquivo(s) ausente(s) para o ID: cjvny0sw6peop0a08dmq1xnc8
[AVISO] Arquivo(s) ausente(s) para o ID: cjvny1w0jpcyb0828enou6i51
[AVISO] Arquivo(s) ausente(s) para o ID: cjvny360qpdht08284w3iivd0
[AVISO] Arquivo(s) ausente(s) para o ID: cjvny51dfpale07952ufne7xc
[AVISO] Arquivo(s) ausente(s) para o ID: cjvnyklt4prx9086649r7s69b
[AVISO] Arquivo(s) ausente(s) para o ID: cjvnzf2a1q09e0838jezgl0ik
[AVISO] Arquivo(s) ausente(s) 

### Pre Processamento

In [3]:
import os
import cv2

def apply_fpls_preprocessing(image_bgr, ksize=3, img_size=(512, 512)):
    # 1. Redimensionamento
    img_resized = cv2.resize(image_bgr, img_size)
    
    # 2. Filtro de Mediana
    img_median = cv2.medianBlur(img_resized, ksize)
    
    # 3. Normalização Min-Max para a faixa [0, 255] em uint8
    img_normalized = cv2.normalize(
        img_median, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U
    )
    
    return img_normalized

def process_and_overwrite_images(base_path="../dataset/preprocess_imgs"):
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"A pasta {base_path} não foi encontrada!")

    image_files = [
        os.path.join(root, file)
        for root, _, files in os.walk(base_path)
        for file in files if file.endswith('.jpeg')
    ]

    print(f"Total de imagens encontradas em preprocess_imgs: {len(image_files)}")

    processed_count = 0
    error_count = 0

    for img_path in image_files:
        img_bgr = cv2.imread(img_path)
        
        if img_bgr is None:
            print(f"[AVISO] Falha ao ler a imagem (pode estar corrompida): {img_path}")
            error_count += 1
            continue
        
        try:
            # Aplica o pré-processamento
            processed_img = apply_fpls_preprocessing(img_bgr)
            
            cv2.imwrite(img_path, processed_img)
            processed_count += 1
        except Exception as e:
            print(f"[ERRO] Falha no processamento de {img_path}: {e}")
            error_count += 1

    print(f"\n[SUCESSO] Imagens pré-processadas e salvas: {processed_count}")
    print(f"[AVISOS] Imagens com falhas: {error_count}")

if __name__ == "__main__":
    process_and_overwrite_images()

Total de imagens encontradas em preprocess_imgs: 1426

[SUCESSO] Imagens pré-processadas e salvas: 1426
[AVISOS] Imagens com falhas: 0


### Verificação de Resolução

In [4]:
from collections import defaultdict
from pathlib import Path
from PIL import Image

caminho_dataset = Path("../dataset/preprocess_imgs")
estatisticas = defaultdict(lambda: defaultdict(int))

for arquivo in caminho_dataset.rglob("*"):
  if arquivo.is_file() and arquivo.suffix == ".jpeg":
    try:
      with Image.open(arquivo) as img:
        largura, altura = img.size
        resolucao = f"{largura}x{altura}"

        nome_pasta = arquivo.relative_to(caminho_dataset).parts[0]

        estatisticas[nome_pasta][resolucao] += 1
    except Exception as e:
      print(f"Não foi possível ler o arquivo {arquivo}: {e}")

print("Resoluções por pasta:")
print("=" * 35)

for pasta, resolucoes in sorted(estatisticas.items()):
  print(f"\n{pasta}")
  for resolucao, qtd in sorted(resolucoes.items()):
    print(f"  - {resolucao} -> {qtd}")

  print("=" * 35)

Resoluções por pasta:

healthy
  - 512x512 -> 726

red_spider_mite
  - 512x512 -> 155

rust
  - 512x512 -> 545
